# MVP v1｜Release Acceptance（只讀）

**Notebook 版本：`MVP-V1-RELEASE-v1`**　｜　Release Candidate pin `09ac0d5`

`Existing Archive → Read-only Evaluator → Comparison → Release Report`

- 來源 run `20261107_081300_9439f8c`（授權 1/1 已耗盡），對照既有 Phase 4-C 評估 `20261107_100344_883d14c`。
- **不執行 `ResearchOnlyRunner.run()`、不重新回測、不安裝 Core、不讀 Holdout、不消耗額度、不修改原始 Archive。**
- 所有邏輯在 `src/strategy_agent/evaluation/release.py`；notebook 只是 thin runner。
- 輸出：`MyDrive/Quant_Research/strategy-research-agent/releases/<YYYYMMDD_HHMMSS>_<rc7>/`（7 個檔案＋不可覆寫）。

**請在全新的 Colab runtime 從 GitHub 開啟並 Run all**；R6（Colab Portability）只有在乾淨 runtime 中整份跑完才會是 PASS。

In [ ]:
# --- 1. 設定（不要修改 pin）---
NOTEBOOK_VERSION = "MVP-V1-RELEASE-v1"
NOTEBOOK_PATH    = "notebooks/MVP_V1_Release_Acceptance.ipynb"
AGENT_REPO = "https://github.com/hh4832/strategy-research-agent.git"
BRANCH     = "release/mvp-v1-candidate"
RC_SHA     = "09ac0d5ac4315e56dcfae44f1deea810d1dc8910"
DRIVE_BASE = "/content/drive/MyDrive/Quant_Research/strategy-research-agent"
SOURCE_ARCHIVE = DRIVE_BASE + "/20261107_081300_9439f8c"
REFERENCE_EVAL = DRIVE_BASE + "/evaluations/20261107_100344_883d14c"
RELEASE_ROOT   = DRIVE_BASE + "/releases"
AGENT_ROOT = "/content/strategy-research-agent-mvp-v1"

import json, os, re, subprocess, sys
from pathlib import Path
assert sys.version_info >= (3, 11), sys.version
CLEAN_RUNTIME = not any(m in sys.modules for m in ("strategy_agent", "market_research_core"))
SOURCE_ARCHIVE, REFERENCE_EVAL, RELEASE_ROOT, AGENT_ROOT = map(Path, (SOURCE_ARCHIVE, REFERENCE_EVAL, RELEASE_ROOT, AGENT_ROOT))
def git(*a, cwd=None):
    r = subprocess.run(["git", *a], cwd=cwd, text=True, capture_output=True)
    if r.returncode: raise RuntimeError("git " + a[0] + " failed: " + r.stderr[-400:])
    return r.stdout.strip()
print(NOTEBOOK_VERSION, "| RC", RC_SHA[:7], "| clean runtime:", CLEAN_RUNTIME)

In [ ]:
# --- 2. Version Sync ---
assert CLEAN_RUNTIME, "STOP: 此 runtime 已載入 strategy_agent / market_research_core；請 Runtime → Restart session 後 Run all"
if not AGENT_ROOT.exists():
    git("clone", "--quiet", AGENT_REPO, str(AGENT_ROOT)); git("checkout", "--quiet", "--detach", RC_SHA, cwd=AGENT_ROOT)
    print("Fresh checkout created")
else:
    print("Existing checkout detected: verifying before reuse")
remote = git("remote", "get-url", "origin", cwd=AGENT_ROOT)
assert remote.rstrip("/").removesuffix(".git") == AGENT_REPO.removesuffix(".git"), "STOP: wrong remote " + remote
head = git("rev-parse", "HEAD", cwd=AGENT_ROOT)
status = git("status", "--porcelain", "--untracked-files=all", cwd=AGENT_ROOT)
git("fetch", "--quiet", "origin", BRANCH, cwd=AGENT_ROOT)
remote_head = git("rev-parse", "FETCH_HEAD", cwd=AGENT_ROOT)
on_branch = subprocess.run(["git", "merge-base", "--is-ancestor", RC_SHA, remote_head], cwd=AGENT_ROOT).returncode == 0
nb = subprocess.run(["git", "show", f"{remote_head}:{NOTEBOOK_PATH}"], cwd=AGENT_ROOT, text=True, capture_output=True)
gh_version = (re.search(r'NOTEBOOK_VERSION = \\"([^\\"]+)\\"', nb.stdout) or [None, None])[1] if nb.returncode == 0 else None
after_pin = git("diff", "--name-only", RC_SHA, remote_head, cwd=AGENT_ROOT).splitlines() if remote_head != RC_SHA else []
SYNC = {"notebook_version": NOTEBOOK_VERSION, "notebook_version_on_github_head": gh_version, "github_remote_head": remote_head,
        "checkout_head": head, "pinned_rc_sha": RC_SHA, "pin_on_branch": on_branch, "working_tree_clean": status == "",
        "files_changed_after_pin": after_pin}
print(json.dumps(SYNC, indent=2))
assert head == RC_SHA, "STOP: checkout HEAD ≠ pin（不自動 checkout/reset；請換 AGENT_ROOT 或重啟 runtime）"
assert status == "", "STOP: checkout 有變更"
assert on_branch, "STOP: pin 不在 GitHub release 分支上"
assert gh_version in (None, NOTEBOOK_VERSION), f"STOP: 這是舊 notebook（{NOTEBOOK_VERSION}），GitHub 上是 {gh_version}；請從 GitHub 重新開啟"
code_after_pin = [p for p in after_pin if not p.startswith(("docs/", "notebooks/"))]
assert not code_after_pin, f"STOP: pin 之後 GitHub 上有程式／治理變更 {code_after_pin}；需要新的 pin"
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(AGENT_ROOT) + "[test,research]"], check=True)
sys.path.insert(0, str(AGENT_ROOT / "src"))
from strategy_agent.evaluation import release as R
assert R.__file__.startswith(str(AGENT_ROOT / "src")), "STOP: release module 不是從 pinned checkout 載入"
assert "market_research_core" not in sys.modules
SYNC["import_source"] = {"release": R.__file__}
print("VERSION SYNC: PASS")

In [ ]:
# --- 3. Regression tests on the pinned RC（subprocess；Core 不安裝 → Core 測試依設計 skip）---
TESTS = R.run_pytest(AGENT_ROOT)
print(json.dumps({k: TESTS[k] for k in ("python", "core", "summary", "counts", "status", "failures")}, indent=2))
assert git("status", "--porcelain", "--untracked-files=all", cwd=AGENT_ROOT) == "", "STOP: tests modified the checkout"

In [ ]:
# --- 4. Drive（只讀）---
from google.colab import drive
drive.mount("/content/drive")
for p, n in ((SOURCE_ARCHIVE, 14), (REFERENCE_EVAL, 8)):
    assert p.is_dir(), "STOP: not found " + str(p)
    names = sorted(x.name for x in p.iterdir()); print(p.name, len(names), names)
    assert len(names) == n, f"STOP: {p} must contain {n} files"

In [ ]:
# --- 5. Read-only evaluation → comparison → release archive ---
RESULT = R.run(SOURCE_ARCHIVE, REFERENCE_EVAL, RELEASE_ROOT, expected_rc_sha=RC_SHA, root=AGENT_ROOT, test_results=TESTS,
               context={"notebook": NOTEBOOK_VERSION, "version_sync": SYNC, "clean_colab_runtime": CLEAN_RUNTIME})
print("VERDICT:", RESULT["verdict"])
print(json.dumps(RESULT["results"], indent=2))
print("Release archive:", RESULT["output_dir"])

In [ ]:
# --- 6. Release report + 下載 ---
import zipfile
OUT = Path(RESULT["output_dir"])
print((OUT / "release_report.md").read_text())
zp = Path("/content") / (OUT.name + "_mvp_v1_release.zip")
with zipfile.ZipFile(zp, "w") as z:
    for p in sorted(OUT.iterdir()): z.write(p, p.name)
from google.colab import files
files.download(str(zp))